In [15]:
!pip install -q transformers accelerate datasets huggingface_hub sentencepiece bitsandbytes
!pip install -q --no-cache-dir "httpx>=0.28.1,<1.0.0"
!pip install -q --no-cache-dir "googletrans==4.0.2"

In [16]:
import os
import time
import json
import torch
import httpx
import googletrans

from tqdm import tqdm
from google.colab import userdata
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    BitsAndBytesConfig,
)
from googletrans import Translator

In [17]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("httpx:", httpx.__version__)
print("googletrans:", googletrans.__version__)

CUDA available: True
GPU: Tesla T4
httpx: 0.28.1
googletrans: 3.4.0


In [18]:
try:
    hf_token = userdata.get("HF_TOKEN")
    print("✅ Đã lấy HF_TOKEN từ Colab Secrets")
except Exception:
    hf_token = None
    print("⚠️ Không tìm thấy HF_TOKEN trong Colab Secrets")

print("📂 Đang tải tập dữ liệu TEST...")

dataset = load_dataset(
    "pqthinh232/HCMUS-Vietnamese-Image-captioning-for-visually-impaired",
    data_files={"test": "test/**"},
    split="test",
    token=hf_token
)

print(f"✅ Thành công! Đã nạp {len(dataset)} sample.")

⚠️ Không tìm thấy HF_TOKEN trong Colab Secrets
📂 Đang tải tập dữ liệu TEST...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/801 [00:00<?, ?it/s]

✅ Thành công! Đã nạp 4000 sample.


In [19]:
MODEL_ID = "Salesforce/blip2-flan-t5-xl"
OUTPUT_FILE = "results_blip2_flan_t5_xl_test.jsonl"

print("🚀 Đang khởi tạo model BLIP-2 FLAN-T5-XL...")

processor = Blip2Processor.from_pretrained(MODEL_ID, use_fast=False)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())

print(f"✅ Model loaded: {MODEL_ID}")
print(f"📦 Tổng số tham số: {total_params / 1e9:.2f} B")

🚀 Đang khởi tạo model BLIP-2 FLAN-T5-XL...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1289 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie language_model.shared.weight to language_model.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model loaded: Salesforce/blip2-flan-t5-xl
📦 Tổng số tham số: 3.94 B


In [20]:
def get_actual_disk_size(model_id):
    model_path = snapshot_download(repo_id=model_id, local_files_only=True)
    total_size = 0

    for dirpath, _, filenames in os.walk(model_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            real_fp = os.path.realpath(fp)
            if os.path.exists(real_fp):
                total_size += os.path.getsize(real_fp)

    return total_size / (1024**3)  # GB

actual_disk_size = get_actual_disk_size(MODEL_ID)
print(f"💾 Disk Size: {actual_disk_size:.2f} GB")

💾 Disk Size: 14.69 GB


In [21]:
async def translate_en_to_vi(text: str, translator: Translator) -> str:
    if not text or text == "[EMPTY_OUTPUT]":
        return "[EMPTY_OUTPUT]"

    try:
        result = await translator.translate(text, src="en", dest="vi")
        translated_text = result.text.strip() if result.text else ""
        return translated_text if translated_text else "[TRANSLATION_ERROR]"
    except Exception:
        return "[TRANSLATION_ERROR]"

In [22]:
prompt_text_en = (
    "Write exactly one fluent sentence describing the scene or obstacles, "
    "and include a short safety suggestion for a visually impaired person."
)

unique_images = {}
print("🔍 Đang trích xuất tên file gốc và lọc ảnh duy nhất...")

for item in dataset:
    full_path = getattr(item["image"], "filename", None)

    if full_path:
        f_name = os.path.basename(full_path)
    else:
        f_name = f"unknown_{len(unique_images)}.jpg"

    if f_name not in unique_images:
        unique_images[f_name] = item["image"]

unique_file_names = sorted(list(unique_images.keys()))
print(f"✅ Thành công! Đã tìm thấy {len(unique_file_names)} ảnh với tên gốc chuẩn.")

🔍 Đang trích xuất tên file gốc và lọc ảnh duy nhất...
✅ Thành công! Đã tìm thấy 800 ảnh với tên gốc chuẩn.


In [23]:
total_time = 0.0
results_data = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"🚀 Bắt đầu xử lý {len(unique_file_names)} ảnh...")

async with Translator(service_urls=["translate.googleapis.com"]) as translator:
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for idx, f_name in enumerate(tqdm(unique_file_names)):
            image = unique_images[f_name].convert("RGB")

            inputs = processor(
                images=image,
                text=prompt_text_en,
                return_tensors="pt"
            )

            # model 8-bit, inputs ảnh/text nên để float16 cho tensor float
            processed_inputs = {}
            for k, v in inputs.items():
                if torch.is_floating_point(v):
                    processed_inputs[k] = v.to(model.device, dtype=torch.float16)
                else:
                    processed_inputs[k] = v.to(model.device)

            t0 = time.time()
            with torch.no_grad():
                generated_ids = model.generate(
                    **processed_inputs,
                    max_new_tokens=40,
                    do_sample=False,
                    num_beams=5,
                    repetition_penalty=1.2,
                    no_repeat_ngram_size=3,
                    early_stopping=True
                )
            t1 = time.time()
            total_time += (t1 - t0)

            caption_en = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0].strip()

            if caption_en == "":
                caption_en = "[EMPTY_OUTPUT]"

            caption_vi = await translate_en_to_vi(caption_en, translator)

            result = {
                "file_name": f_name,
                "prediction_en": caption_en,
                "prediction": caption_vi
            }

            results_data.append(result)
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

            if idx < 3:
                print(f"\n--- DEBUG {f_name} ---")
                print("EN :", repr(caption_en))
                print("VI :", repr(caption_vi))

print(f"✅ Đã lưu kết quả tại: {OUTPUT_FILE}")

🚀 Bắt đầu xử lý 800 ảnh...


  0%|          | 0/800 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
  0%|          | 1/800 [00:06<1:32:42,  6.96s/it]


--- DEBUG 00003.jpg ---
EN : 'driving on a country road with lots of trees'
VI : 'lái xe trên con đường quê có nhiều cây xanh'


  0%|          | 2/800 [00:13<1:32:52,  6.98s/it]


--- DEBUG 00009.jpg ---
EN : "there's a lot of people walking down the street"
VI : 'có rất nhiều người đi bộ trên đường'


  0%|          | 3/800 [00:18<1:18:51,  5.94s/it]


--- DEBUG 00014.jpg ---
EN : "there's a lot of cars on the road"
VI : 'có rất nhiều xe ô tô trên đường'


100%|██████████| 800/800 [46:32<00:00,  3.49s/it]

✅ Đã lưu kết quả tại: results_blip2_flan_t5_xl_test.jsonl


In [24]:
avg_time = total_time / len(unique_file_names)

if torch.cuda.is_available():
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
else:
    peak_vram = 0.0

print("\n" + "=" * 40)
print("📊 EFFICIENCY METRICS")
print(f"- Model: {MODEL_ID}")
print(f"- Params: {total_params / 1e9:.2f} B")
print(f"- Disk Size: {actual_disk_size:.2f} GB")
print(f"- Time/Img: {avg_time:.4f} s")
print(f"- Peak VRAM: {peak_vram:.2f} GB")
print(f"- Results saved at: {OUTPUT_FILE}")
print("=" * 40)


📊 EFFICIENCY METRICS
- Model: Salesforce/blip2-flan-t5-xl
- Params: 3.94 B
- Disk Size: 14.69 GB
- Time/Img: 2.6549 s
- Peak VRAM: 4.59 GB
- Results saved at: results_blip2_flan_t5_xl_test.jsonl


In [25]:
print("📄 Xem trước 10 kết quả đầu tiên:\n")

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(json.loads(line))

📄 Xem trước 10 kết quả đầu tiên:

{'file_name': '00003.jpg', 'prediction_en': 'driving on a country road with lots of trees', 'prediction': 'lái xe trên con đường quê có nhiều cây xanh'}
{'file_name': '00009.jpg', 'prediction_en': "there's a lot of people walking down the street", 'prediction': 'có rất nhiều người đi bộ trên đường'}
{'file_name': '00014.jpg', 'prediction_en': "there's a lot of cars on the road", 'prediction': 'có rất nhiều xe ô tô trên đường'}
{'file_name': '00027.jpg', 'prediction_en': 'cars are driving down a road with trees on either side', 'prediction': 'ô tô đang chạy trên con đường có cây cối hai bên'}
{'file_name': '00030.jpg', 'prediction_en': 'there are motorcycles on a busy street', 'prediction': 'có những chiếc xe máy trên một con phố đông đúc'}
{'file_name': '00048.jpg', 'prediction_en': "there's a lot of cars on the road", 'prediction': 'có rất nhiều xe ô tô trên đường'}
{'file_name': '00061.jpg', 'prediction_en': "there's a street with cars on it", 'predi

In [26]:
empty_count = 0
translation_error_count = 0
total_count = 0

with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        total_count += 1

        if item["prediction_en"] == "[EMPTY_OUTPUT]":
            empty_count += 1

        if item["prediction"] == "[TRANSLATION_ERROR]":
            translation_error_count += 1

print("Tổng số ảnh:", total_count)
print("Số output EN rỗng:", empty_count)
print("Số lỗi dịch:", translation_error_count)
print("Tỉ lệ output EN rỗng:", empty_count / total_count if total_count else 0)
print("Tỉ lệ lỗi dịch:", translation_error_count / total_count if total_count else 0)

Tổng số ảnh: 800
Số output EN rỗng: 0
Số lỗi dịch: 0
Tỉ lệ output EN rỗng: 0.0
Tỉ lệ lỗi dịch: 0.0
